# DSC@UIT 2026 — Task 1: LegalIR — v5 BGE-Gemma 3B

Bản thử nghiệm có cổng an toàn: dùng bi-encoder v3B đã fine-tune (0.6B), thay reranker cũ bằng `BAAI/bge-reranker-v2-gemma` (~3B), tổng bảo thủ ~3.6B < 4B. Notebook chỉ tạo submission nếu validation vượt v3B.


> **Đây không phải v4 và không train lại reranker v3B.** Notebook tái sử dụng model/embedding v3B từ dataset, sau đó dùng một reranker multilingual lớn hơn. Chỉ chạy một lần với RUN_PHASE=2.


---
## Stage 0 — Config & môi trường

In [1]:
#@title Config
import os, sys, glob

# ---- Môi trường: Colab / Kaggle / local -------------------------------------
# Kiểm tra Kaggle TRƯỚC vì Kaggle không có /content, còn Colab thì luôn có.
# Trên Windows '/content' được hiểu là 'C:\content' nên phải chặn bằng os.name.
def _detect_env():
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE') or os.environ.get('KAGGLE_URL_BASE'):
        return 'kaggle'
    if os.name == 'posix' and os.path.isdir('/kaggle/input'):
        return 'kaggle'
    if 'google.colab' in sys.modules or os.environ.get('COLAB_RELEASE_TAG'):
        return 'colab'
    if os.name == 'posix' and os.path.isdir('/content'):
        return 'colab'
    return 'local'

ENV       = _detect_env()
ON_COLAB  = ENV == 'colab'
ON_KAGGLE = ENV == 'kaggle'
PROFILE   = 'v3b_ft_reranker_improved'   # cache + tên file nộp riêng cho bản này.

# ---- Chạy Kaggle thành 2 Version --------------------------------------------
# Version 1: build cache + train reranker + lưu checkpoint, rồi kết thúc sạch.
# Version 2: attach output Version 1, load checkpoint và chỉ chạy val/public.
# Logic train/inference, seed và hyperparameter giữ nguyên; chỉ bỏ các bước đã xong.
RUN_PHASE = 2   # FAST SUBMIT: chỉ inference từ artifact Version 1
assert RUN_PHASE == 2, 'FAST SUBMIT chỉ chạy RUN_PHASE=2'
# CHẠY SONG SONG HAI BẢN: trên Kaggle mỗi notebook có /kaggle/working
# riêng nên an toàn. Trên Colab thì KHÔNG — cả hai mount cùng My Drive;
# phải để PROFILE khác nhau VÀ cap khác nhau (xem _CAPTAG dưới), nếu
# không hai session ghi đè shard .npy của nhau và làm hỏng cả hai.

# ---- GPU ---------------------------------------------------------------------
# '' = dùng hết GPU thấy được (Kaggle: 2×T4 -> Stage 5 và 7 chia việc cho cả hai).
#      Đây là giá trị nên dùng: Stage 8 tự ép mình về 1 GPU trong lúc train (xem
#      cell đó), nên không còn phải hy sinh tốc độ encode để fine-tune được nữa.
# '0' = buộc 1 GPU cho toàn bộ notebook (chỉ cần khi debug).
# Đổi biến này BẮT BUỘC phải Restart runtime rồi chạy lại từ đầu (torch đọc
# CUDA_VISIBLE_DEVICES một lần duy nhất lúc khởi tạo).
VISIBLE_GPUS = ''   #@param ['', '0', '1', '0,1'] {allow-input: true}
if VISIBLE_GPUS:
    os.environ['CUDA_VISIBLE_DEVICES'] = VISIBLE_GPUS

# ---- Nơi GHI artifact --------------------------------------------------------
# Colab : My Drive CỦA BẠN (không ghi vào thư mục chia sẻ của nhóm: ~1.5GB và
#         người khác có thể xoá mất).
# Kaggle: /kaggle/working — 20GB, và là thư mục DUY NHẤT còn lại sau "Save Version".
#         /kaggle/temp không vào output nên dùng làm đĩa tạm.
if ON_COLAB:
    ART_ROOT = '/content/drive/MyDrive/DSC2026_Task1'
    SCRATCH  = '/content/scratch'
elif ON_KAGGLE:
    ART_ROOT = '/kaggle/working/DSC2026_Task1'
    SCRATCH  = '/kaggle/temp/scratch'
else:
    ART_ROOT = './work'
    SCRATCH  = './scratch'
ART = f'{ART_ROOT}/artifacts'

# ---- Cache CHỈ ĐỌC từ session trước (Kaggle) ---------------------------------
# Kaggle không có Drive: cách tái dùng embedding giữa các session là "Save Version"
# rồi attach output đó lại làm Dataset ở lần sau (xem Stage 11 ở cuối notebook).
#
# DÒ chứ không ghim pattern. Kaggle mount dataset ở độ sâu khác nhau tuỳ cách
# attach: /kaggle/input/<slug>/... khi attach từ "Your Work", nhưng
# /kaggle/input/datasets/<user>/<slug>/... khi attach dataset của người khác.
# Cộng thêm tiền tố DSC2026_Task1/ nằm sẵn trong zip là thành 4 tầng — pattern
# cứng trượt mà KHÔNG báo lỗi, notebook chỉ lặng lẽ encode lại toàn corpus.
# Nên: quét /kaggle/input tìm mọi thư mục tên 'artifacts', và chỉ nhận cái nào
# thật sự chứa artifact của pipeline này.
def _find_art_in(root='/kaggle/input', max_depth=6):
    found = []
    if not os.path.isdir(root):
        return found
    base = root.rstrip('/').count('/')
    for dirpath, dirnames, _ in os.walk(root):
        if dirpath.count('/') - base >= max_depth:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames
                       if not d.startswith('.') and d != '__MACOSX']
        if 'artifacts' in dirnames:
            p = os.path.join(dirpath, 'artifacts')
            # phải có dấu vết của pipeline, tránh nhận nhầm thư mục trùng tên
            if os.path.exists(f'{p}/corpus.parquet') or os.path.isdir(f'{p}/emb'):
                found.append(p)
    return sorted(dict.fromkeys(found))

# Dataset Phase 1 đã được đóng gói riêng; ưu tiên nó trước mọi Notebook Output khác.
V3B_ART_HINTS = [
    '/kaggle/input/datasets/kitnehi1211/v3b-finetune-reranker-improve/DSC2026_Task1/artifacts',
    '/kaggle/input/v3b-finetune-reranker-improve/DSC2026_Task1/artifacts',
    '/kaggle/input/datasets/kitnehi1211/v3b-finetune-reranker-improve/artifacts',
    '/kaggle/input/v3b-finetune-reranker-improve/artifacts',
]
ART_IN = ([p for p in V3B_ART_HINTS if os.path.isdir(p)] + _find_art_in()) if ON_KAGGLE else []
ART_IN = list(dict.fromkeys(ART_IN))

def cache_path(rel):
    """Tìm cache theo thứ tự: ART (ghi được) -> ART_IN (chỉ đọc). None nếu chưa có."""
    p = f'{ART}/{rel}'
    if os.path.exists(p):
        return p
    for d in ART_IN:
        q = f'{d}/{rel}'
        if os.path.exists(q):
            return q
    return None

def first_existing(paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None

# ---- Dataset đã attach trên Kaggle -------------------------------------------
# Kaggle mount dataset theo /kaggle/input/<slug>, nhưng khi attach kèm tên chủ sở
# hữu thì đường dẫn thành /kaggle/input/datasets/<user>/<slug>. Ghim cả hai dạng;
# không trúng dạng nào thì cell sau tự dò trong /kaggle/input.
KAGGLE_DATA_HINTS = [
    '/kaggle/input/datasets/kitnehi1211/uit-task1-publictest/LegalR - Public Test',
    '/kaggle/input/uit-task1-publictest/LegalR - Public Test',
]
KAGGLE_SCORING_HINTS = [
    '/kaggle/input/datasets/kitnehi1211/uit-task1-scoring/Scoring-Program-Task-LegalIR',
    '/kaggle/input/uit-task1-scoring/Scoring-Program-Task-LegalIR',
]
# scoring.py THẬT của BTC -> Stage 3 dùng để đối chiếu hàm chấm điểm tự viết
SCORING_DIR = first_existing(KAGGLE_SCORING_HINTS + [
    './Scoring-Program-Task-LegalIR', '../Scoring-Program-Task-LegalIR'])

# ---- Nơi ĐỌC dữ liệu: đường dẫn cố định theo môi trường ---------------------
# Đường dẫn nào không tồn tại thì cell sau tự dò.
DATA_FILES = ('train.json', 'public-official.json')
if ON_COLAB:
    RAW = '/content/drive/MyDrive/UIT DS Challenge/Task 1/LegalR - Public Test'
elif ON_KAGGLE:
    RAW = first_existing(KAGGLE_DATA_HINTS)
else:
    RAW = './LegalR - Public Test'
CTX_DIR = f'{SCRATCH}/contexts'   # nơi giải nén zip (đĩa tạm, KHÔNG vào Drive/output)

# ---- Chunking ----------------------------------------------------------------
CHUNK_CHARS        = 3000   # phù hợp context dài của Vietnamese_Embedding_v2
CHUNK_OVERLAP      = 500
MAX_CHUNKS_PER_DOC = 2500
# Đo trên dữ liệu thật: cap=150 cắt mất nội dung của 6.3% số lần gold doc
# xuất hiện, trong khi chỉ 41/8532 doc bị cap và bỏ cap chỉ thêm 3.5%
# chunk (154k -> 160k). Gold doc dài gấp ~3 lần trung bình corpus
# (p50 64k vs 22k ký tự) nên đây là mất mát thật.
#
# Cap PHẢI nằm trong tên cache. TAG/CHUNKS_REL bản v2 chỉ chứa
# CHUNK_CHARS, nên đổi cap mà không đổi tên là cache_path() trả về
# parquet + shard embedding CŨ trong im lặng. Giữ nguyên tên khi cap=150
# để cache của các lần chạy trước vẫn dùng lại được.
CHUNKER_VER = 't1'
# Đổi khi logic chunk_doc() ở Stage 2 đổi -> tên cache đổi theo. Thiếu cái này thì
# sửa chunker mà cache_path() vẫn trả về parquet + shard của chunker CŨ, im lặng.
_CAPTAG = (('' if MAX_CHUNKS_PER_DOC == 150 else f'_m{MAX_CHUNKS_PER_DOC}')
           + (f'_{CHUNKER_VER}' if CHUNKER_VER else ''))

# ---- Retrieval ---------------------------------------------------------------
BM25_K1, BM25_B = 0.9, 0.4  # tuned cho document dài
CAND_CHUNKS     = 1000      # số chunk lấy ra trước khi gộp về document
CAND_DOCS       = 100       # số doc mỗi nhánh đưa vào RRF
RRF_K           = 60
RERANK_TOP      = 30       # số doc đưa vào cross-encoder
TOPK_SUBMIT     = 5         # RÀNG BUỘC CỨNG của BTC

# Cách dùng điểm của cross-encoder. ĐỪNG đoán — Stage 7 in cả ba, chọn theo val:
#   'rrf'     : RRF giữa thứ tự hybrid và thứ tự rerank. Reranker sai một câu thì
#               phiếu của retriever vẫn giữ được gold trong top-5 -> an toàn nhất.
#   'replace' : tin hẳn reranker (cách cũ). Chỉ chọn khi val cho thấy nó thắng rõ.
#   'none'    : bỏ rerank. Đúng là một lựa chọn: ở lần chạy thử trên corpus rút
#               gọn, hybrid RRF (recall@5 0.919) còn cao hơn 'replace' (0.858).
RERANK_MODE     = 'rrf'   #@param ['rrf', 'replace', 'none']

# ---- Ba tham số thêm sau khi đọc log profile v2 ------------------------------
# Log v2 nói rất rõ:
#   dense              recall@5 = 0.8996  <- TỐT NHẤT
#   hybrid RRF         recall@5 = 0.8699  <- tụt 3 điểm so với dense
#   hybrid + rerank    recall@5 = 0.8973  <- vẫn chưa vượt nổi dense đơn thuần
# Nguyên nhân: RRF cho BM25 (recall@5 = 0.6868) một phiếu NGANG BẰNG dense.
# Và dense recall@100 = 0.9897 -> khâu tìm ứng viên hết chỗ cải thiện; toàn bộ
# dư địa còn lại nằm ở XẾP HẠNG trong top-30 (trần ~0.975).

# Base đưa vào cross-encoder và dùng cho submission.
# Stage 7 đo CẢ HAI rồi in ra cái nào thắng — sửa biến này theo bảng đó.
RERANK_BASE = 'dense'   #@param ['dense', 'hybrid']

# Trọng số RRF cho (bm25, dense). (1,1) là RRF cổ điển, chỉ đúng khi hai kênh
# mạnh ngang nhau. Stage 6 sweep vài giá trị rồi chọn theo val.
RRF_WEIGHTS = (1.0, 3.0)

# Số chunk mỗi doc đưa cho cross-encoder; điểm doc = MAX trên các chunk đó.
# Trước đây = 1: với nhiều chunk/doc, reranker chỉ
# được đọc 1/150 nội dung của doc -> chọn nhầm chunk đại diện là gold chết oan.
# Chi phí GPU tăng tuyến tính (30 doc x 3 chunk = 90 cặp/query).
RERANK_CHUNKS_PER_DOC = 2   # bản B: cố định = config lúc train reranker

# ---- Models (phải đăng ký với BTC trước khi dùng) ----------------------------
BIENCODER = 'AITeamVN/Vietnamese_Embedding_v2'    # 0.6B
RERANKER  = 'BAAI/bge-reranker-v2-gemma'        # ~3B; thay reranker cũ, tổng ~3.6B < 4B
MAX_SEQ   = 1024
def format_dense_queries(texts):
    return list(texts)
def format_dense_passages(texts):
    return list(texts)
FT_BIENCODER_REL = f'models/biencoder-ft-{PROFILE}'
if RUN_PHASE == 1:
    FT_BIENCODER_PATH = f'{ART}/{FT_BIENCODER_REL}'
else:
    FT_BIENCODER_PATH = cache_path(FT_BIENCODER_REL)
    assert FT_BIENCODER_PATH and os.path.isdir(FT_BIENCODER_PATH), (
        'Không thấy bi-encoder đã fine-tune trong output Version 1. '
        'Hãy attach đúng Notebook Output trước khi chạy RUN_PHASE=2.')
# Stage 5 encode bằng checkpoint nào lúc BẮT ĐẦU chạy notebook.
#   False -> checkpoint gốc. Đây là giá trị đúng cho lần chạy đầu: Stage 8 cần
#            retrieval của model gốc để mine hard negative.
#   Stage 8 train xong sẽ TỰ bật biến này lên True + encode lại toàn corpus,
#   nên không cần sửa tay rồi chạy lại notebook từ đầu nữa.
USE_FINETUNED = (RUN_PHASE == 2)
DO_FINETUNE   = (RUN_PHASE == 1)  # train đúng một lần ở Version 1
# TẮT ở cả hai bản v3. recall@100 = 0.9897: retriever đã xong việc, dư địa
# nằm ở XẾP HẠNG. Fine-tune bi-encoder tốn thêm một lần encode lại toàn
# corpus mà tối ưu đúng cái đã hết chỗ tối ưu.

# ---- Eval --------------------------------------------------------------------
N_VAL = 1000
SEED  = 42

# ---- Batch: các số này là batch cho MỖI GPU ---------------------------------
# T4 Colab 16GB và T4 Kaggle 15GB gần như bằng nhau -> giữ nguyên batch, tốc độ
# tăng là do chia việc cho 2 GPU chứ không phải do batch to hơn. Kaggle P100
# (1×16GB) cũng chạy đúng cấu hình này.
ENC_SHARD     = 20000   # encode theo shard, checkpoint từng shard ra đĩa
ENC_BATCH     = 16
RERANK_MAXLEN = 512      # bản B: passage ngắn tập trung, = maxlen lúc train
RERANK_BATCH  = 8
RERANK_FP16   = True     # reranker chạy fp16: ~2x nhanh, chênh lệch điểm không đáng kể

print('ENV =', ENV, '| PROFILE =', PROFILE, '| RUN_PHASE =', RUN_PHASE, '| RERANK_MODE =', RERANK_MODE)
print('ART      =', ART)
print('SCRATCH  =', SCRATCH)
if RAW:
    print('RAW      =', RAW)
if SCORING_DIR:
    print('scoring.py BTC:', SCORING_DIR)
if ART_IN:
    print('cache chỉ đọc:', ART_IN)


ENV = kaggle | PROFILE = v3b_ft_reranker_improved | RUN_PHASE = 2 | RERANK_MODE = rrf
ART      = /kaggle/working/DSC2026_Task1/artifacts
SCRATCH  = /kaggle/temp/scratch
RAW      = /kaggle/input/datasets/kitnehi1211/uit-task1-publictest/LegalR - Public Test
scoring.py BTC: /kaggle/input/datasets/kitnehi1211/uit-task1-scoring/Scoring-Program-Task-LegalIR
cache chỉ đọc: ['/kaggle/input/datasets/kitnehi1211/v3b-finetune-reranker-improve/DSC2026_Task1/artifacts']


In [2]:
#@title Mount Drive (Colab) + cài dependency + liệt kê GPU
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# Pin cùng một bản Sentence Transformers trên cả Colab lẫn Kaggle.
ST_WANT = '5.4.0'  # tương thích cả legal E5 và Vietnamese cross-encoder
try:
    import sentence_transformers as _st
    ST_HAVE = _st.__version__
except ImportError:
    ST_HAVE = None

if ST_HAVE != ST_WANT:
    !pip -q install sentence-transformers=={ST_WANT} pyarrow 2>&1 | tail -1
    import importlib, sentence_transformers as _st
    ST_HAVE = importlib.reload(_st).__version__

print('sentence-transformers', ST_HAVE, '(bản đã test:', ST_WANT + ')')

import os
for d in (ART, f'{ART}/emb', f'{ART}/models', SCRATCH):
    os.makedirs(d, exist_ok=True)

import torch
N_GPU   = torch.cuda.device_count()
DEVICES = [f'cuda:{i}' for i in range(N_GPU)] or ['cpu']
DEV     = DEVICES[0]

print('torch', torch.__version__, '| GPU thấy được:', N_GPU)
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f'  cuda:{i}  {p.name}  {p.total_memory/1e9:.1f} GB')
if N_GPU > 1:
    print(f'-> Stage 5 (encode corpus) và Stage 7 (rerank) sẽ chia việc cho {N_GPU} GPU')
    print('   Stage 8 (fine-tune) tự ép về 1 GPU trong lúc train, không cần sửa gì.')
elif N_GPU == 1:
    print('-> 1 GPU: pipeline chạy tuần tự như bình thường')
else:
    print('-> KHÔNG có GPU. Stage 1-4 (BM25) vẫn chạy được và nên chạy ở session CPU '
          'để tiết kiệm quota; Stage 5 trở đi bắt buộc phải có GPU.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.8/570.8 kB 6.7 MB/s eta 0:00:00
sentence-transformers 5.4.0 (bản đã test: 5.4.0)
torch 2.10.0+cu128 | GPU thấy được: 2
  cuda:0  Tesla T4  15.6 GB
  cuda:1  Tesla T4  15.6 GB
-> Stage 5 (encode corpus) và Stage 7 (rerank) sẽ chia việc cho 2 GPU
   Stage 8 (fine-tune) tự ép về 1 GPU trong lúc train, không cần sửa gì.


In [3]:
#@title Dò dữ liệu + giải nén selected-contexts.zip
# Ưu tiên RAW cố định ở Config; không có thì dò (Kaggle: /kaggle/input/<dataset>).
# Giải nén vào đĩa tạm chứ KHÔNG vào Drive / /kaggle/working: 8.532 file nhỏ ghi
# lên Drive mất hàng giờ, và đếm vào giới hạn 20GB output của Kaggle.
import os, glob, zipfile, time

def find_data_dir(roots, max_depth=6):
    """Thư mục đầu tiên chứa đủ DATA_FILES."""
    for root in roots:
        if not os.path.isdir(root):
            continue
        base = root.rstrip('/').count('/')
        for dirpath, dirnames, filenames in os.walk(root):
            if dirpath.count('/') - base >= max_depth:
                dirnames[:] = []
                continue
            dirnames[:] = [d for d in dirnames
                           if not d.startswith('.') and d != '__MACOSX']
            if all(f in filenames for f in DATA_FILES):
                return dirpath
    return None

def data_roots():
    if ON_COLAB:
        return ['/content/drive/MyDrive',
                '/content/drive/.shortcut-targets-by-id',   # nơi shortcut trỏ tới
                '/content/drive/Shareddrives',
                '/content']
    if ON_KAGGLE:
        return ['/kaggle/input']                            # mọi dataset đã attach
    return ['.', '..']

def n_ctx(d):
    return len(glob.glob(f'{d}/**/context_*.json', recursive=True)) if d else 0

if RAW is None or not all(os.path.exists(f'{RAW}/{f}') for f in DATA_FILES):
    print('đường dẫn cố định không dùng được -> đang dò...', flush=True)
    RAW = find_data_dir(data_roots())

assert RAW, (
    'KHÔNG tìm thấy dữ liệu (cần train.json + public-official.json).\n\n'
    'Colab: thư mục "UIT DS Challenge" thường ở "Shared with me" mà Colab không\n'
    '  mount phần đó. Sửa 1 lần: drive.google.com -> Shared with me -> chuột phải\n'
    '  "UIT DS Challenge" -> Organise -> Add shortcut to Drive -> My Drive.\n\n'
    'Kaggle: phải Add Input dataset chứa dữ liệu Task 1 vào notebook\n'
    '  (Notebook -> Add Input -> Datasets). Kiểm tra: !ls /kaggle/input\n\n'
    'Hoặc gán tay: RAW = "<đường dẫn tới thư mục LegalR - Public Test>"')

print('RAW =', RAW)
CTX_ZIP = f'{RAW}/selected-contexts.zip'

if cache_path('corpus.parquet'):
    print('corpus.parquet đã có -> bỏ qua giải nén')
elif n_ctx(RAW):
    CTX_DIR = RAW                     # dataset đã bung sẵn (hay gặp trên Kaggle)
    print(f'đã bung sẵn trong RAW: {n_ctx(CTX_DIR)} file')
elif n_ctx(CTX_DIR):
    print(f'đã giải nén trước đó: {n_ctx(CTX_DIR)} file')
else:
    assert os.path.exists(CTX_ZIP), (
        f'Không thấy {CTX_ZIP} và cũng không thấy file context_*.json nào.\n'
        'Colab: kiểm tra shortcut "UIT DS Challenge" trong My Drive.\n'
        'Kaggle: dataset đã attach phải chứa selected-contexts (zip hoặc đã bung).')
    os.makedirs(CTX_DIR, exist_ok=True)
    t0 = time.time()
    with zipfile.ZipFile(CTX_ZIP) as z:
        print(f'{len(z.namelist())} entry, đang giải nén...', flush=True)
        z.extractall(CTX_DIR)
    print(f'xong {time.time()-t0:.0f}s -> {n_ctx(CTX_DIR)} file context')

if not cache_path('corpus.parquet'):
    n = n_ctx(CTX_DIR)
    assert n == 8532, f'mong đợi 8532 file context, thấy {n} — zip/dataset có thể thiếu'
    print('OK: đủ 8532 văn bản')


RAW = /kaggle/input/datasets/kitnehi1211/uit-task1-publictest/LegalR - Public Test
corpus.parquet đã có -> bỏ qua giải nén


---
## Stage 1 — Build corpus

Đọc 8.532 file `context_*.json` → **một** file parquet.

Hai vấn đề dữ liệu được xử lý ở đây:
- Zip giải nén **lồng hai lớp** (`selected-contexts/selected-contexts/...`) → glob đệ quy trong `CTX_DIR`.
- **1.125/8.532 văn bản (13%) thiếu field `name`** → lấy lại tiêu đề từ slug trong `link`.
  Tiêu đề là tín hiệu rất mạnh (chứa loại văn bản + chủ đề + năm), bỏ 13% là mất điểm thật.

Cell này chỉ đọc `CTX_DIR` (đã bung ở cell Giải nén phía trên) — không tự dò thư mục khác.

In [4]:
#@title Build corpus.parquet
import json, re, glob, os, time
import pandas as pd

CORPUS_PQ = f'{ART}/corpus.parquet'

def _title_from_link(link):
    slug = (link or '').rstrip('/').rsplit('/', 1)[-1]
    slug = re.sub(r'\.aspx$', '', slug, flags=re.I)
    return slug.replace('-', ' ').strip()

def build_corpus():
    # Chỉ đọc CTX_DIR — cell "Giải nén selected-contexts.zip" ở trên đã bung sẵn.
    # Zip lồng hai lớp (selected-contexts/selected-contexts/...) nên glob đệ quy.
    files = sorted(glob.glob(f'{CTX_DIR}/**/context_*.json', recursive=True))
    assert files, f'{CTX_DIR} trống — hãy chạy lại cell "Giải nén selected-contexts.zip".'
    print(f'{len(files)} file context', flush=True)

    rows, n_noname = [], 0
    for i, fp in enumerate(files):
        with open(fp, encoding='utf-8') as f:
            d = json.load(f)
        title = (d.get('name') or '').replace('-', ' ').strip()
        if not title:
            n_noname += 1
            title = _title_from_link(d.get('link', ''))
        # gộp mọi whitespace: passage có rất nhiều khoảng trắng / newline rác.
        # Làm sớm để regex tách 'Điều N' ở Stage 2 hoạt động ổn định.
        passage = re.sub(r'\s+', ' ', d.get('passage') or '').strip()
        rows.append({'doc_id': str(d['id']), 'title': title,
                     'link': d.get('link', ''), 'passage': passage})
        if (i + 1) % 2000 == 0:
            print(f'  {i+1}/{len(files)}', flush=True)

    df = pd.DataFrame(rows).drop_duplicates('doc_id').reset_index(drop=True)
    print(f'-> {len(df)} doc | thiếu name (đã vá từ link): {n_noname}')
    df.to_parquet(CORPUS_PQ, index=False)
    return df

t0 = time.time()
_pq = cache_path('corpus.parquet')      # ART trước, rồi tới dataset chỉ đọc (Kaggle)
if _pq:
    corpus = pd.read_parquet(_pq)
    print(f'load cache: {len(corpus)} doc <- {_pq}')
else:
    corpus = build_corpus()

print('%.1fs' % (time.time() - t0))
print('passage chars: mean %d | p50 %d | max %d' % (
    corpus.passage.str.len().mean(), corpus.passage.str.len().median(),
    corpus.passage.str.len().max()))
corpus.head(3)[['doc_id', 'title']]

load cache: 8532 doc <- /kaggle/input/datasets/kitnehi1211/v3b-finetune-reranker-improve/DSC2026_Task1/artifacts/corpus.parquet
4.5s
passage chars: mean 39451 | p50 22089 | max 5787180


,doc_id,title
0,100050,TCVN 13268 1 2021 Bao ve thuc vat dieu tra sin...
1,100062,Thong tu 17 2022 TT BGTVT sua doi Thong tu 12 ...
2,100109,Quyet dinh 569 QD TTg 2022 Chien luoc phat tri...


In [5]:
#@title Load train / public + val split cố định
import json, random

with open(f'{RAW}/train.json', encoding='utf-8') as f:
    train_raw = json.load(f)
with open(f'{RAW}/public-official.json', encoding='utf-8') as f:
    public_raw = json.load(f)

# answer có thể là int hoặc str tuỳ file -> luôn ép str
QUESTIONS = {qid: v['question'] for qid, v in train_raw.items()}
GOLD = {qid: set(map(str, v['answer'])) for qid, v in train_raw.items()}
PUBLIC = {qid: v['question'] for qid, v in public_raw.items()}

# Split cố định theo seed + sorted key -> mọi session cho cùng một val set.
# Đây là điều kiện bắt buộc để so sánh các thí nghiệm với nhau.
all_qids = sorted(train_raw.keys())
random.Random(SEED).shuffle(all_qids)
VAL_QIDS   = all_qids[:N_VAL]
TRAIN_QIDS = all_qids[N_VAL:]

print(f'train {len(TRAIN_QIDS)} | val {len(VAL_QIDS)} | public {len(PUBLIC)}')

# kiểm tra gold có nằm trong corpus không
corpus_ids = set(corpus.doc_id)
missing = {g for q in all_qids for g in GOLD[q]} - corpus_ids
print('gold doc thiếu trong corpus:', len(missing))

train 6000 | val 1000 | public 1000
gold doc thiếu trong corpus: 0


---
## Stage 2 — Chunking theo cấu trúc pháp quy

Passage trung bình **41.000 ký tự** mà bi-encoder chỉ nhận ~256 token (~1.000 ký tự).
Không chunk là không dùng được dense retrieval.

Ba quyết định thiết kế ở đây:
- **Cắt theo `Điều N`** thay vì cắt cứng theo độ dài — ranh giới ngữ nghĩa của văn bản
  pháp luật nằm đúng ở đó. Điều ngắn thì gộp lại, Điều dài thì cắt tiếp có overlap.
- **Prepend tiêu đề vào mọi chunk.** Câu hỏi rất ngắn (trung bình 20 từ) nên tiêu đề
  văn bản là tín hiệu định chủ đề cực mạnh.
- **Chunk đầu tiên = tiêu đề + phần mở đầu**, nơi chứa số hiệu, cơ quan ban hành, trích yếu.
- **Cap `MAX_CHUNKS_PER_DOC`**: nếu không, vài văn bản 6M ký tự sẽ chiếm phần lớn index
  và bóp méo điểm `max` khi gộp chunk→doc.

In [6]:
#@title Build chunks.parquet
import re, os, time
import pandas as pd

CHUNKS_REL = f'chunks_c{CHUNK_CHARS}{_CAPTAG}.parquet'
CHUNKS_PQ  = f'{ART}/{CHUNKS_REL}'

DIEU_RE = re.compile(r'(?=Điều\s+\d+\s*[\.\:])')

def _split_long(s, size, overlap):
    """Cắt s thành các đoạn <= size ký tự, ưu tiên cắt ở ranh giới câu."""
    out, i = [], 0
    while i < len(s):
        j = min(i + size, len(s))
        if j < len(s):
            # lùi về dấu chấm gần nhất trong 200 ký tự cuối cho khỏi cắt giữa câu
            cut = s.rfind('. ', i + size - 200, j)
            if cut > i:
                j = cut + 1
        out.append(s[i:j].strip())
        if j >= len(s):
            break
        i = max(j - overlap, i + 1)
    return [c for c in out if c]

def chunk_doc(title, passage):
    """-> list các đoạn text (chưa gắn tiêu đề)."""
    pieces = [p.strip() for p in DIEU_RE.split(passage) if p.strip()]
    if not pieces:
        pieces = [passage]

    merged = []
    buf = ''
    for p in pieces:
        if len(p) > CHUNK_CHARS:
            if buf:
                merged.append(buf); buf = ''
            merged.extend(_split_long(p, CHUNK_CHARS, CHUNK_OVERLAP))
        elif len(buf) + len(p) + 1 <= CHUNK_CHARS:
            buf = f'{buf} {p}'.strip()
        else:
            merged.append(buf); buf = p
    if buf:
        merged.append(buf)

    # chunk mở đầu: chứa số hiệu / cơ quan ban hành / trích yếu
    head = passage[:CHUNK_CHARS].strip()
    if head and (not merged or merged[0][:200] != head[:200]):
        merged.insert(0, head)
    return merged[:MAX_CHUNKS_PER_DOC]

def build_chunks(corpus):
    rows, n_titleonly = [], 0
    for n, r in enumerate(corpus.itertuples(index=False)):
        cs = chunk_doc(r.title, r.passage)
        if not cs:
            # 20 văn bản có `passage` RỖNG và cũng không có field `name` (tiêu đề
            # được vá từ slug trong `link` ở trên). chunk_doc() chỉ chunk passage
            # nên chúng nhận 0 chunk -> dense retrieval KHÔNG BAO GIỜ truy hồi
            # được. 6/20 doc này từng là gold, và 9 câu hỏi train mất trắng vì mọi
            # gold của chúng nằm trong nhóm đó. Phát một chunk chỉ chứa tiêu đề:
            # tiêu đề vá từ slug link có loại văn bản + chủ đề + năm, đủ để khớp ở
            # mức chủ đề — hơn hẳn không có gì.
            cs, n_titleonly = [''], n_titleonly + 1
        for c in cs:
            # tiêu đề đi kèm mọi chunk -> chunk lẻ vẫn biết mình thuộc văn bản nào
            rows.append((r.doc_id, f'{r.title}. {c}'.strip()))
        if (n + 1) % 1000 == 0:
            print(f'  {n+1}/{len(corpus)} doc -> {len(rows)} chunk', flush=True)
    if n_titleonly:
        print(f'-> {n_titleonly} doc passage rỗng: đã phát chunk chỉ-tiêu-đề')
    df = pd.DataFrame(rows, columns=['doc_id', 'text'])
    df.to_parquet(CHUNKS_PQ, index=False)
    return df

t0 = time.time()
_cpq = cache_path(CHUNKS_REL)
if _cpq:
    chunks = pd.read_parquet(_cpq)
    print('load cache <-', _cpq)
else:
    chunks = build_chunks(corpus)
print('%d chunk | %.1fs' % (len(chunks), time.time() - t0))

# map chunk -> doc dạng số nguyên, dùng suốt phần sau
DOC_IDS = corpus.doc_id.tolist()
DOC2IX  = {d: i for i, d in enumerate(DOC_IDS)}
import numpy as np
CHUNK2DOC = np.array([DOC2IX[d] for d in chunks.doc_id], dtype=np.int32)
print('chunk/doc trung bình: %.1f' % (len(chunks) / len(corpus)))
chunks.text.str.len().describe()[['mean', '50%', 'max']]

load cache <- /kaggle/input/datasets/kitnehi1211/v3b-finetune-reranker-improve/DSC2026_Task1/artifacts/chunks_c3000_m2500_t1.parquet
151034 chunk | 4.7s
chunk/doc trung bình: 17.7


mean    2537.283923
50%     2912.000000
max     3147.000000
Name: text, dtype: float64

---
## Stage 3 — Hạ tầng đánh giá

**Làm cái này trước khi tối ưu bất cứ gì.** `public-official.json` không có đáp án và chỉ
được nộp 10 lần/ngày, nên mọi quyết định kỹ thuật phải ra từ val set này — miễn phí, không
giới hạn số lần thử.

Công thức theo đúng BTC: `Recall = mean_q |gold ∩ pred| / |gold|` với `pred` cắt ở 5 doc.

Luôn in cả `@5`, `@20`, `@100` vì chúng chẩn đoán hai bệnh khác nhau:
- `@100` thấp → **retriever** yếu (văn bản đúng không vào nổi danh sách ứng viên)
- `@100` cao mà `@5` thấp → **ranking** yếu (tìm ra rồi nhưng không đẩy lên top)

In [7]:
#@title Metrics + experiment log
import numpy as np, pandas as pd, os, datetime

def official_score(sub, truth):
    """Bản sao CHÍNH XÁC của scoring.py mà BTC dùng trên Codabench.

    sub: {qid: {'answer': [id]}} — đúng định dạng file nộp
    truth: {qid: [id]}

    Ba chi tiết trong code của BTC mà đọc mô tả không thấy:
    1. Recall KHÔNG cắt top-5. Nó chỉ *chặn* ở len<=5: quá 5 id -> câu đó 0 điểm.
       Nộp đúng 5 id là tối ưu cho Recall.
    2. Số lượng key phải khớp tuyệt đối, lệch là raise Exception -> TOÀN BỘ bài nộp
       lỗi, không phải chỉ mất điểm câu đó.
    3. Thiếu key -> len(None) -> TypeError -> cũng sập toàn bài.
    """
    y_pred = {k: v['answer'] for k, v in sub.items()}
    if len(y_pred) != len(truth):
        raise Exception('Samples in predict not match with reference')
    ok = lambda k: 0 < len(y_pred[k]) <= TOPK_SUBMIT
    recall = np.mean([len(set(truth[k]) & set(y_pred[k])) / len(truth[k])
                      if ok(k) else 0 for k in truth])
    precision = np.mean([len(set(truth[k]) & set(y_pred[k])) / len(y_pred[k])
                         if ok(k) else 0 for k in y_pred])
    return {'recall': float(recall), 'precision': float(precision)}

def evaluate(preds, gold, ks=(5, 10, 20, 30, 50, 100)):
    """preds: {qid: [doc_id đã xếp hạng]} -> recall@k chẩn đoán + điểm chính thức.

    recall@20/@100 KHÔNG phải metric của cuộc thi — chúng để chẩn đoán:
      @100 thấp        -> retriever yếu (văn bản đúng không vào nổi danh sách)
      @100 cao, @5 thấp -> ranking yếu (tìm ra rồi mà không đẩy lên top)

    recall@RERANK_TOP (mặc định @30) là **trần cứng** của Stage 7: cross-encoder
    chỉ được nhìn top-RERANK_TOP, doc không lọt vào đó thì không cách nào cứu.
    Đọc `@30` và `@5` cạnh nhau -> biết ngay còn bao nhiêu điểm đang bỏ trên bàn.
    """
    out = {}
    for k in ks:
        out[f'recall@{k}'] = float(np.mean([
            len(gold[q] & set(preds.get(q, [])[:k])) / len(gold[q]) for q in gold]))
    # điểm chính thức: dựng đúng shape file nộp rồi chấm bằng code của BTC
    sub = {q: {'answer': list(preds.get(q, [])[:TOPK_SUBMIT])} for q in gold}
    off = official_score(sub, {q: list(gold[q]) for q in gold})
    out['OFFICIAL_recall'] = off['recall']
    out['OFFICIAL_precision'] = off['precision']
    return out

def show(name, preds, gold=None, qids=None):
    gold = gold or {q: GOLD[q] for q in (qids or VAL_QIDS)}
    m = evaluate(preds, gold)
    print(name.ljust(28), ' '.join(f'{k}={v:.4f}' for k, v in m.items()))
    return m

LOG_CSV = f'{ART}/experiments.csv'

def log_exp(name, metrics, note=''):
    """Ghi lại mọi thí nghiệm. Bảng này là thứ bạn cần để chọn cấu hình cho private
    test và để viết paper nếu vào top 10 — đừng bỏ qua."""
    row = {'time': datetime.datetime.now().isoformat(timespec='seconds'),
           'profile': PROFILE, 'run': name, 'note': note, **metrics}
    df = pd.DataFrame([row])
    if os.path.exists(LOG_CSV):
        df = pd.concat([pd.read_csv(LOG_CSV), df], ignore_index=True)
    df.to_csv(LOG_CSV, index=False)
    return df.tail(5)

print('ok')


# ---- Đối chiếu official_score() với scoring.py THẬT của BTC ------------------
# Dataset uit-task1-scoring có sẵn mã chấm điểm Codabench -> nạp trực tiếp
# eval_retrieval() của BTC và so với bản chép tay ở trên trên vài trường hợp
# biên. Nếu lệch thì mọi con số OFFICIAL_* trong notebook đều vô nghĩa.
import importlib.util, glob as _glob

def _load_btc_eval():
    cands = ([f'{SCORING_DIR}/scoring.py'] if SCORING_DIR else []) + \
            sorted(_glob.glob('/kaggle/input/**/Scoring-Program-Task-LegalIR/scoring.py',
                              recursive=True))
    for p in cands:
        if not os.path.exists(p):
            continue
        spec = importlib.util.spec_from_file_location('btc_scoring', p)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)          # main() có guard __main__ -> import an toàn
        return mod.eval_retrieval, p
    return None, None

_btc_eval, _btc_path = _load_btc_eval()
if _btc_eval is None:
    print('không thấy scoring.py của BTC -> bỏ qua đối chiếu '
          '(Kaggle: Add Input dataset uit-task1-scoring)')
else:
    _truth = {'a': ['1'], 'b': ['2', '3'], 'c': ['4']}
    _cases = {
        'đúng 1/1':        {'a': ['1'], 'b': ['2', '3'], 'c': ['4']},
        'nộp 5 id':        {'a': ['1', 'x', 'y', 'z', 'w'], 'b': ['2', '3', 'x', 'y', 'z'],
                            'c': ['q', 'w', 'e', 'r', 't']},
        'nộp 6 id -> 0':   {'a': ['1', 'x', 'y', 'z', 'w', 'v'], 'b': ['2'], 'c': ['4']},
    }
    _ok = True
    for _name, _p in _cases.items():
        _mine = official_score({k: {'answer': v} for k, v in _p.items()}, _truth)
        _theirs = _btc_eval({k: {'answer': v} for k, v in _p.items()}, _truth)
        _same = (abs(_mine['recall'] - _theirs['recall']) < 1e-9 and
                 abs(_mine['precision'] - _theirs['precision']) < 1e-9)
        _ok &= _same
        print(f'  {_name:16s} recall={_theirs["recall"]:.4f} '
              f'precision={_theirs["precision"]:.4f}  {"khớp" if _same else "LỆCH!"}')
    assert _ok, 'official_score() lệch so với scoring.py của BTC — sửa trước khi tin số liệu'
    print('official_score() khớp scoring.py của BTC <-', _btc_path)


ok
  đúng 1/1         recall=1.0000 precision=1.0000  khớp
  nộp 5 id         recall=0.6667 precision=0.2000  khớp
  nộp 6 id -> 0    recall=0.5000 precision=0.6667  khớp
official_score() khớp scoring.py của BTC <- /kaggle/input/datasets/kitnehi1211/uit-task1-scoring/Scoring-Program-Task-LegalIR/scoring.py


In [8]:
#@title Gộp chunk -> doc bằng MAX, rồi lấy top-N doc
def chunks_to_docs(chunk_scores, chunk_top_ix, topn=CAND_DOCS):
    """Điểm doc = MAX điểm các chunk của nó (không phải mean).

    Gold là "văn bản CHỨA thông tin cần thiết" -> một chunk khớp là đủ. Lấy mean sẽ
    phạt oan các văn bản dài mà chỉ một Điều liên quan.
    """
    best = {}
    for ix, s in zip(chunk_top_ix, chunk_scores):
        d = CHUNK2DOC[ix]
        if s > best.get(d, -1e30):
            best[d] = s
    ranked = sorted(best.items(), key=lambda x: -x[1])[:topn]
    return [DOC_IDS[d] for d, _ in ranked], [s for _, s in ranked]

print('ok')

ok


---
## Stage 4 — BM25 (chunk-level)

Không cần GPU — **chạy stage này trong session CPU** để tiết kiệm quota GPU, đó là tài
nguyên khan hiếm nhất của bạn.

Cài đặt BM25 dạng sparse matrix (`W @ q`) thay vì `rank_bm25`: với ~500k chunk thì
`rank_bm25` chậm tới mức không dùng được, còn sparse matmul chấm 1 query mất vài chục ms.

BM25 bù đúng chỗ dense yếu: thuật ngữ pháp lý hiếm (*"Kiểm ngư viên trung cấp"*,
*"hành lang an toàn kỹ thuật anten"*) mà embedding thường trượt hẳn.

In [9]:
#@title BM25 sparse
import numpy as np, scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer

class SparseBM25:
    """BM25 Okapi tiền tính thành ma trận trọng số -> truy vấn = 1 phép matmul sparse."""

    def __init__(self, k1=BM25_K1, b=BM25_B, min_df=3, max_features=400_000):
        self.k1, self.b = k1, b
        self.vec = CountVectorizer(lowercase=True, min_df=min_df,
                                   max_features=max_features, dtype=np.int32)

    def fit(self, texts):
        X = self.vec.fit_transform(texts).tocsc()
        n, _ = X.shape
        dl = np.asarray(X.sum(axis=1)).ravel()
        avgdl = dl.mean()
        df = np.diff(X.indptr)
        idf = np.log(1 + (n - df + 0.5) / (df + 0.5)).astype(np.float32)

        X = X.tocoo()
        tf = X.data.astype(np.float32)
        denom = tf + self.k1 * (1 - self.b + self.b * dl[X.row] / avgdl)
        data = idf[X.col] * tf * (self.k1 + 1) / denom
        self.W = sp.csr_matrix((data, (X.row, X.col)), shape=(n, len(idf),))
        self.W = self.W.T.tocsr()      # (vocab, n_chunk) -> query nhanh hơn
        return self

    def search(self, queries, topk=CAND_CHUNKS):
        Q = self.vec.transform(queries)
        Q.data[:] = 1.0               # query dùng binary term presence
        S = (Q.astype(np.float32) @ self.W).toarray()   # (nq, n_chunk)
        k = min(topk, S.shape[1])
        ix = np.argpartition(-S, k - 1, axis=1)[:, :k]
        srt = np.take_along_axis(S, ix, 1).argsort(axis=1)[:, ::-1]
        ix = np.take_along_axis(ix, srt, 1)
        return ix, np.take_along_axis(S, ix, 1)

import time, pickle
t0 = time.time()
bm25 = SparseBM25().fit(chunks.text.tolist())
print('BM25 fit: %.1fs | vocab %d' % (time.time() - t0, bm25.W.shape[0]))

BM25 fit: 49.4s | vocab 48842


In [10]:
#@title Đánh giá BM25 trên val
def retrieve_bm25(qids, qtext, batch=64):
    preds = {}
    for i in range(0, len(qids), batch):
        bq, bt = qids[i:i+batch], qtext[i:i+batch]
        ix, sc = bm25.search(bt)
        for q, row_ix, row_sc in zip(bq, ix, sc):
            preds[q] = chunks_to_docs(row_sc, row_ix)[0]
    return preds

t0 = time.time()
bm25_val = retrieve_bm25(VAL_QIDS, [QUESTIONS[q] for q in VAL_QIDS])
print('%.1fs' % (time.time() - t0))
m = show('BM25 chunk-level', bm25_val)
log_exp('bm25', m, f'chunk={CHUNK_CHARS} k1={BM25_K1} b={BM25_B}')

8.8s
BM25 chunk-level             recall@5=0.6917 recall@10=0.7903 recall@20=0.8597 recall@30=0.8937 recall@50=0.9179 recall@100=0.9484 OFFICIAL_recall=0.6917 OFFICIAL_precision=0.1486


,time,profile,run,note,recall@5,recall@10,recall@20,recall@30,recall@50,recall@100,OFFICIAL_recall,OFFICIAL_precision
0,2026-09-06T14:29:44,v3b_ft_reranker_improved,bm25,chunk=3000 k1=0.9 b=0.4,0.69175,0.79025,0.859667,0.893667,0.917917,0.948417,0.69175,0.1486


---
## Stage 5 — Dense retrieval

Bước nặng duy nhất. **Corpus là cố định và dùng chung cho cả train lẫn public test** →
encode **một lần**, tái dùng cho mọi thí nghiệm sau. Đây là điểm tiết kiệm lớn nhất trên Colab.

Encode theo **shard 50k chunk, ghi từng shard ra Drive ngay**. Session Colab chết ở phút
thứ 50 mà không có shard là mất trắng cả tiếng GPU; có shard thì chỉ chạy lại phần thiếu.

Vector lưu **fp16** (~500k × 768 × 2B ≈ 750MB) — vừa RAM, vừa VRAM T4, không cần faiss:
`q @ emb.T` trên GPU đã đủ nhanh và khỏi phải build index.

In [11]:
#@title Encode corpus (đa GPU, sharded, resume được)
import numpy as np, os, glob, time, re, torch
from sentence_transformers import SentenceTransformer

def set_dense_model(model_path):
    """Chọn checkpoint cho nhánh dense + trỏ EMB_DIR sang cache của đúng checkpoint đó.

    Tách thành hàm vì Stage 8 gọi lại nó sau khi fine-tune: model đổi thì embedding
    cũ vô giá trị, phải đổi cả nơi ghi lẫn nơi đọc chứ không chỉ đổi mỗi model.
    """
    global MODEL_PATH, TAG, EMB_REL, EMB_DIR, EMB_DIRS
    MODEL_PATH = model_path
    _is_ft = os.path.basename(os.path.normpath(MODEL_PATH)) == f'biencoder-ft-{PROFILE}'
    _model_key = f'biencoder_ft_{PROFILE}' if _is_ft else re.sub(r'[^a-zA-Z0-9]+', '_', MODEL_PATH)
    TAG     = _model_key + f'_c{CHUNK_CHARS}{_CAPTAG}'
    EMB_REL = f'emb/{TAG}'
    EMB_DIR = f'{ART}/{EMB_REL}'          # nơi GHI
    os.makedirs(EMB_DIR, exist_ok=True)
    # nơi ĐỌC: thư mục ghi + mọi cache chỉ đọc (Kaggle: output session trước attach lại)
    EMB_DIRS = [EMB_DIR] + [f'{d}/{EMB_REL}' for d in ART_IN if os.path.isdir(f'{d}/{EMB_REL}')]
    print('model    ->', MODEL_PATH)
    print('ghi      ->', EMB_DIR)
    if len(EMB_DIRS) > 1:
        print('đọc thêm <-', EMB_DIRS[1:])

def _shard_file(a):
    """shard đã encode ở BẤT KỲ cache nào -> path, chưa có -> None."""
    for d in EMB_DIRS:
        p = f'{d}/s_{a:08d}.npy'
        if os.path.exists(p):
            return p
    return None

def _l2(v):
    """normalize tay: encode_multi_process ở một số bản ST không nhận
    normalize_embeddings, làm ở đây thì mọi bản đều cho ra vector cùng thang."""
    v = np.asarray(v, dtype=np.float32)
    v /= np.linalg.norm(v, axis=1, keepdims=True) + 1e-12
    return v

def encode_corpus():
    texts  = format_dense_passages(chunks.text.tolist())
    shards = [(i, min(i + ENC_SHARD, len(texts))) for i in range(0, len(texts), ENC_SHARD)]
    todo   = [(a, b) for a, b in shards if _shard_file(a) is None]

    if todo:
        model = SentenceTransformer(MODEL_PATH, device=DEV)
        model.max_seq_length = MAX_SEQ
        if torch.cuda.is_available():
            model.half()

        # >1 GPU: mỗi GPU một process con, ST tự chia đều sentence cho các process.
        # Đây là chỗ ăn tốc độ lớn nhất của Kaggle 2×GPU — encode là bước nặng nhất.
        pool = None
        if len(DEVICES) > 1:
            try:
                pool = model.start_multi_process_pool(target_devices=DEVICES)
                print(f'multi-GPU pool: {DEVICES}', flush=True)
            except Exception as e:
                print('không mở được multi-process pool (%s) -> chạy 1 GPU' % e, flush=True)
                pool = None

        print(f'{len(todo)}/{len(shards)} shard cần encode', flush=True)
        try:
            for a, b in todo:
                t0 = time.time()
                if pool is not None:
                    v = model.encode_multi_process(
                        texts[a:b], pool, batch_size=ENC_BATCH,
                        chunk_size=max(500, (b - a) // (4 * len(DEVICES))))
                else:
                    v = model.encode(texts[a:b], batch_size=ENC_BATCH,
                                     convert_to_numpy=True, show_progress_bar=False)
                np.save(f'{EMB_DIR}/s_{a:08d}.npy', _l2(v).astype(np.float16))
                print(f'  shard {a}-{b}: {time.time()-t0:.0f}s', flush=True)
        finally:
            if pool is not None:
                model.stop_multi_process_pool(pool)
            del model
            torch.cuda.empty_cache()

    return np.concatenate([np.load(_shard_file(a)) for a, _ in shards])

set_dense_model(FT_BIENCODER_PATH if USE_FINETUNED else BIENCODER)
t0  = time.time()
EMB = encode_corpus()
assert len(EMB) == len(chunks), f'emb {len(EMB)} != chunks {len(chunks)} — xoá {EMB_DIR} và encode lại'
print('EMB', EMB.shape, EMB.dtype, '| %.1f MB | %.1fs' % (EMB.nbytes / 1e6, time.time() - t0))


model    -> /kaggle/input/datasets/kitnehi1211/v3b-finetune-reranker-improve/DSC2026_Task1/artifacts/models/biencoder-ft-v3b_ft_reranker_improved
ghi      -> /kaggle/working/DSC2026_Task1/artifacts/emb/biencoder_ft_v3b_ft_reranker_improved_c3000_m2500_t1
đọc thêm <- ['/kaggle/input/datasets/kitnehi1211/v3b-finetune-reranker-improve/DSC2026_Task1/artifacts/emb/biencoder_ft_v3b_ft_reranker_improved_c3000_m2500_t1']
EMB (151034, 1024) float16 | 309.3 MB | 2.6s


In [12]:
#@title Dense search trên GPU
import torch, numpy as np
from sentence_transformers import SentenceTransformer

# EMB fp16 chỉ vài trăm MB nên để trọn trên 1 GPU: q @ EMB.T là một phép matmul,
# chia cho 2 GPU chỉ thêm đồng bộ chứ không nhanh hơn đáng kể. GPU còn lại được
# dùng thật sự ở Stage 5 (encode) và Stage 7 (rerank) — hai bước nặng.
EMB_T  = None
qmodel = None

def free_dense():
    """Nhả EMB_T + query encoder khỏi VRAM. Stage 8 gọi trước khi train: chỗ này
    đang giữ vài GB, để nguyên thì fine-tune gần như chắc chắn OOM."""
    global EMB_T, qmodel
    EMB_T = qmodel = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def load_query_model():
    """Đưa EMB lên GPU + nạp encoder cho query theo MODEL_PATH hiện tại.
    Gọi lại được — Stage 8 dùng để chuyển sang model vừa fine-tune."""
    global EMB_T, qmodel
    free_dense()
    EMB_T = torch.from_numpy(EMB).to(DEV)                   # (N, dim) fp16
    qmodel = SentenceTransformer(MODEL_PATH, device=DEV)
    qmodel.max_seq_length = MAX_SEQ
    if DEV.startswith('cuda'):
        qmodel.half()

def dense_search(qtext, topk=CAND_CHUNKS, batch=32):
    """-> (indices, scores) của top chunk cho mỗi query."""
    ixs, scs = [], []
    for i in range(0, len(qtext), batch):
        q = qmodel.encode(format_dense_queries(qtext[i:i+batch]), batch_size=batch, normalize_embeddings=True,
                          convert_to_tensor=True, show_progress_bar=False).to(EMB_T.dtype)
        s = q @ EMB_T.T                                     # cosine (đã normalize)
        v, ix = torch.topk(s, min(topk, s.shape[1]), dim=1)
        ixs.append(ix.cpu().numpy()); scs.append(v.float().cpu().numpy())
    return np.concatenate(ixs), np.concatenate(scs)

def retrieve_dense(qids, qtext):
    ix, sc = dense_search(qtext)
    return {q: chunks_to_docs(s, i)[0] for q, i, s in zip(qids, ix, sc)}

load_query_model()
t0 = time.time()
val_text = [QUESTIONS[q] for q in VAL_QIDS]
dense_val = retrieve_dense(VAL_QIDS, val_text)
print('%.1fs' % (time.time() - t0))
m = show('dense' + ('-ft' if USE_FINETUNED else ''), dense_val)
log_exp('dense_ft' if USE_FINETUNED else 'dense', m, MODEL_PATH)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

2.8s
dense-ft                     recall@5=0.9283 recall@10=0.9606 recall@20=0.9794 recall@30=0.9827 recall@50=0.9849 recall@100=0.9879 OFFICIAL_recall=0.9283 OFFICIAL_precision=0.2002


,time,profile,run,note,recall@5,recall@10,recall@20,recall@30,recall@50,recall@100,OFFICIAL_recall,OFFICIAL_precision
0,2026-09-06T14:29:44,v3b_ft_reranker_improved,bm25,chunk=3000 k1=0.9 b=0.4,0.691750,0.790250,0.859667,0.893667,0.917917,0.948417,0.691750,0.1486
1,2026-09-06T14:30:14,v3b_ft_reranker_improved,dense_ft,/kaggle/input/datasets/kitnehi1211/v3b-finetun...,0.928333,0.960583,0.979417,0.982667,0.984917,0.987917,0.928333,0.2002


---
## Stage 6 — Hybrid fusion (RRF **có trọng số**)

Điểm BM25 và điểm cosine nằm trên hai thang hoàn toàn khác nhau nên chuẩn hoá chúng
luôn mong manh — RRF chỉ dùng **hạng**, bền hơn nhiều:

`score(d) = Σ wᵢ / (K + rankᵢ(d))`

### Vì sao có `wᵢ` — bài học từ profile v2

| | recall@5 |
|---|---|
| BM25 | 0.6868 |
| dense | **0.8996** |
| hybrid RRF w=(1,1) | 0.8699 |

RRF cổ điển cho mọi kênh **một phiếu bằng nhau**. BM25 yếu hơn dense 21 điểm mà vẫn
được bỏ phiếu ngang cơ → fusion tụt 3 điểm so với chỉ dùng dense. Đây không phải lỗi
cài đặt, đó là RRF đang làm đúng thứ nó được bảo làm.

Cell dưới sweep `w = (1,1) … (1,5)` **và** cả phương án bỏ hẳn BM25, rồi chọn theo val.
Sweep này thuần CPU, không đụng tới quota GPU — không có lý do gì để đoán.

> BM25 vẫn được giữ lại vì nó phục vụ hai việc khác: chọn chunk đại diện cho doc ở
> Stage 7 (doc lọt vào ứng viên nhờ từ khoá hiếm thường không có chunk nào trong top
> dense) và mine hard negative ở Stage 8.


In [13]:
#@title RRF fuse (có trọng số) + sweep chọn trọng số theo val
def rrf(*rankings, k=RRF_K, topn=CAND_DOCS, weights=None):
    """Mỗi ranking: {qid: [doc_id]} -> {qid: [doc_id] đã fuse}.

    `weights` là điểm mới so với bản trước. RRF cổ điển cho mọi kênh một phiếu
    bằng nhau, và đó chính là lý do hybrid v2 (0.8699) THUA dense đơn thuần
    (0.8996): BM25 chỉ đạt recall@5 = 0.6868 nhưng vẫn được quyền bỏ phiếu ngang
    dense. Kênh yếu hơn phải có phiếu nhẹ hơn.
    """
    ws = list(weights) if weights else [1.0] * len(rankings)
    assert len(ws) == len(rankings)
    out = {}
    for q in rankings[0]:
        s = {}
        for w, r in zip(ws, rankings):
            for rank, d in enumerate(r.get(q, [])):
                s[d] = s.get(d, 0.0) + w / (k + rank + 1)
        out[q] = [d for d, _ in sorted(s.items(), key=lambda x: -x[1])[:topn]]
    return out


# Sweep trọng số: rẻ (thuần CPU, không đụng GPU) nên đo hết rồi chọn, đừng đoán.
# dense đơn thuần cũng là một ứng viên hợp lệ — và theo log v2 nó đang dẫn đầu.
_cands = {'dense (bỏ hẳn BM25)': dense_val}
for _w in [(1, 1), (1, 2), (1, 3), (1, 5)]:
    _cands[f'hybrid RRF w={_w}'] = rrf(bm25_val, dense_val, weights=_w)

_scores = {}
for _n, _p in _cands.items():
    _scores[_n] = show(_n, _p)

_best_name = max(_scores, key=lambda n: (_scores[n]['OFFICIAL_recall'],
                                         _scores[n]['OFFICIAL_precision']))
print(f'\n-> val chọn: {_best_name}  (recall@5 = {_scores[_best_name]["recall@5"]:.4f})')

# hybrid_val = nhánh RRF tốt nhất (KHÔNG phải dense — Stage 7 so hai base riêng).
_best_w = max([w for w in [(1, 1), (1, 2), (1, 3), (1, 5)]],
              key=lambda w: _scores[f'hybrid RRF w={w}']['OFFICIAL_recall'])
if tuple(RRF_WEIGHTS) != tuple(_best_w):
    print(f'   RRF_WEIGHTS ở Stage 0 đang là {tuple(RRF_WEIGHTS)}, val thích {_best_w}'
          ' -> sửa Stage 0 cho khớp trước khi chạy Stage 9.')
hybrid_val = _cands[f'hybrid RRF w={_best_w}']
log_exp('hybrid_rrf_w', _scores[f'hybrid RRF w={_best_w}'], f'K={RRF_K} w={_best_w}')


dense (bỏ hẳn BM25)          recall@5=0.9283 recall@10=0.9606 recall@20=0.9794 recall@30=0.9827 recall@50=0.9849 recall@100=0.9879 OFFICIAL_recall=0.9283 OFFICIAL_precision=0.2002
hybrid RRF w=(1, 1)          recall@5=0.8863 recall@10=0.9443 recall@20=0.9691 recall@30=0.9792 recall@50=0.9877 recall@100=0.9894 OFFICIAL_recall=0.8863 OFFICIAL_precision=0.1900
hybrid RRF w=(1, 2)          recall@5=0.9137 recall@10=0.9618 recall@20=0.9774 recall@30=0.9837 recall@50=0.9862 recall@100=0.9904 OFFICIAL_recall=0.9137 OFFICIAL_precision=0.1966
hybrid RRF w=(1, 3)          recall@5=0.9260 recall@10=0.9648 recall@20=0.9809 recall@30=0.9837 recall@50=0.9864 recall@100=0.9879 OFFICIAL_recall=0.9260 OFFICIAL_precision=0.1996
hybrid RRF w=(1, 5)          recall@5=0.9307 recall@10=0.9673 recall@20=0.9807 recall@30=0.9847 recall@50=0.9849 recall@100=0.9879 OFFICIAL_recall=0.9307 OFFICIAL_precision=0.2008

-> val chọn: hybrid RRF w=(1, 5)  (recall@5 = 0.9307)
   RRF_WEIGHTS ở Stage 0 đang là (1.0, 3.0), 

,time,profile,run,note,recall@5,recall@10,recall@20,recall@30,recall@50,recall@100,OFFICIAL_recall,OFFICIAL_precision
0,2026-09-06T14:29:44,v3b_ft_reranker_improved,bm25,chunk=3000 k1=0.9 b=0.4,0.691750,0.790250,0.859667,0.893667,0.917917,0.948417,0.691750,0.1486
1,2026-09-06T14:30:14,v3b_ft_reranker_improved,dense_ft,/kaggle/input/datasets/kitnehi1211/v3b-finetun...,0.928333,0.960583,0.979417,0.982667,0.984917,0.987917,0.928333,0.2002
2,2026-09-06T14:30:14,v3b_ft_reranker_improved,hybrid_rrf_w,"K=60 w=(1, 5)",0.930750,0.967333,0.980667,0.984667,0.984917,0.987917,0.930750,0.2008


---

## Stage v5 — reranker 3B với validation gate

Không dùng reranker v3B 0.6B trong pipeline này vì 0.6 + 0.6 + 3.0 = 4.2B, vượt giới hạn. Quantization không làm giảm số tham số được tính.


In [14]:
#@title Khai báo model và kiểm tra đăng ký với BTC
import os, sys, subprocess, importlib.util

BGE_RERANKER = 'BAAI/bge-reranker-v2-gemma'
MODEL_APPROVED_BY_BTC = True  #@param {type:'boolean'}
assert MODEL_APPROVED_BY_BTC, (
    'DỪNG ĐỂ ĐÚNG LUẬT: hãy đăng ký/kiểm tra BAAI/bge-reranker-v2-gemma với BTC, '
    'sau khi được chấp nhận mới đổi MODEL_APPROVED_BY_BTC=True.')

# FlagEmbedding 1.3.5 lỗi với Transformers 5.x vì còn import
# is_torch_fx_available đã bị xóa. Bản 1.4.2 đã có compatibility fix.
from importlib.metadata import version as _pkg_version, PackageNotFoundError
FLAG_WANT = '1.4.2'
try:
    FLAG_HAVE = _pkg_version('FlagEmbedding')
except PackageNotFoundError:
    FLAG_HAVE = None
if FLAG_HAVE != FLAG_WANT:
    assert 'FlagEmbedding' not in sys.modules, (
        'FlagEmbedding cũ đã được import. Restart Session rồi Run All từ đầu.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                    'FlagEmbedding==' + FLAG_WANT], check=True)
print('FlagEmbedding', _pkg_version('FlagEmbedding'), '(bản yêu cầu:', FLAG_WANT + ')')

assert RUN_PHASE == 2, 'v5 chỉ dùng artifacts v3B đã có: giữ RUN_PHASE=2'
assert len(DEVICES) >= 1 and DEVICES[0].startswith('cuda'), 'Bắt buộc bật GPU'
print('MODEL MỚI:', BGE_RERANKER)
print('Ngân sách bảo thủ: bi-encoder 0.6B + reranker 3.0B = 3.6B < 4B')
print('Thiết bị:', DEVICES)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.5/250.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.8/947.8 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 52.6 MB/s eta 0:00:00
FlagEmbedding 1.4.2 (bản yêu cầu: 1.4.2)
MODEL MỚI: BAAI/bge-reranker-v2-gemma
Ngân sách bảo thủ: bi-encoder 0.6B + reranker 3.0B = 3.6B < 4B
Thiết bị: ['cuda:0', 'cuda:1']


In [15]:
#@title Chuẩn bị top-30 dense và ghép 2 chunk tốt nhất mỗi document
import numpy as np, torch, time

assert np.all(np.diff(CHUNK2DOC) >= 0), 'CHUNK2DOC phải tăng dần'
_doc_ar = np.arange(len(DOC_IDS))
DOC_CHUNK_LO = np.searchsorted(CHUNK2DOC, _doc_ar, 'left')
DOC_CHUNK_HI = np.searchsorted(CHUNK2DOC, _doc_ar, 'right')
CHUNK_TEXTS = chunks.text.astype(str).tolist()

def select_doc_chunks(qids, qtext, candidates, batch=32):
    """Chọn 2 chunk dense tốt nhất trong TOÀN BỘ chunk của doc, không chỉ top-1000."""
    qids, out = list(qids), {}
    for i0 in range(0, len(qtext), batch):
        qv = qmodel.encode(format_dense_queries(qtext[i0:i0+batch]), batch_size=batch,
                           normalize_embeddings=True, convert_to_tensor=True,
                           show_progress_bar=False).to(EMB_T.dtype)
        scores = (qv @ EMB_T.T).float().cpu().numpy()
        for r, q in enumerate(qids[i0:i0+batch]):
            chosen = {}
            for d in candidates[q][:RERANK_TOP]:
                di = DOC2IX[d]
                lo, hi = int(DOC_CHUNK_LO[di]), int(DOC_CHUNK_HI[di])
                if hi <= lo:
                    continue
                local = scores[r, lo:hi]
                kk = min(RERANK_CHUNKS_PER_DOC, hi-lo)
                ix = np.argpartition(-local, kk-1)[:kk]
                ix = ix[np.argsort(-local[ix])]
                chosen[d] = [lo + int(j) for j in ix]
            out[q] = chosen
    return out

pub_qids = list(PUBLIC.keys())
pub_text = [PUBLIC[q] for q in pub_qids]
t0 = time.time()
pub_dense = retrieve_dense(pub_qids, pub_text)
VAL_CHUNK = select_doc_chunks(VAL_QIDS, val_text, dense_val)
PUB_CHUNK = select_doc_chunks(pub_qids, pub_text, pub_dense)
print('Chuẩn bị candidate/chunk val+public: %.1f phút' % ((time.time()-t0)/60))
print('Số cặp dự kiến:', sum(map(len, VAL_CHUNK.values())), 'val +',
      sum(map(len, PUB_CHUNK.values())), 'public')

# Dense retrieval đã hoàn tất cho cả val/public; nhả VRAM trước khi nạp hai model 3B.
free_dense()


Chuẩn bị candidate/chunk val+public: 0.1 phút
Số cặp dự kiến: 30000 val + 29976 public


In [16]:
#@title Nạp BGE-Gemma trên từng GPU và smoke test
import gc, threading, numpy as np, torch
from FlagEmbedding import FlagLLMReranker

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

probe = [['Mức phạt đối với hành vi vi phạm là bao nhiêu?',
          'Văn bản quy định mức phạt tiền đối với hành vi vi phạm.'],
         ['Mức phạt đối với hành vi vi phạm là bao nhiêu?',
          'Hôm nay thời tiết đẹp và có nắng.']]
BGE_MODELS = {}
for dev in DEVICES:
    print('loading', BGE_RERANKER, '->', dev, flush=True)
    model = FlagLLMReranker(
        BGE_RERANKER,
        query_max_length=256,
        max_length=512,
        batch_size=4,
        use_fp16=True,
        devices=[dev]
    )
    # Gọi ngay để model float32 trên CPU được half() rồi chuyển sang GPU trước khi
    # nạp bản kế tiếp; tránh giữ đồng thời hai bản float32 (~20 GB RAM CPU).
    probe_score = np.asarray(model.compute_score(
        probe, batch_size=1, normalize=False), dtype=float).reshape(-1)
    assert len(probe_score) == 2 and np.isfinite(probe_score).all(), probe_score
    assert np.ptp(probe_score) > 1e-6, 'Reranker trả điểm hằng số'
    print(dev, 'smoke scores:', probe_score.tolist())
    BGE_MODELS[dev] = model


loading BAAI/bge-reranker-v2-gemma -> cuda:0


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/555 [00:00<?, ?B/s]

...

100%|██████████| 2/2 [00:00<00:00, 19.14it/s]

cuda:0 smoke scores: [5.1171875, -5.484375]
loading BAAI/bge-reranker-v2-gemma -> cuda:1


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 18.32it/s]

cuda:1 smoke scores: [5.1171875, -5.484375]


In [17]:
#@title Chấm điểm song song trên các GPU
import time, threading
from collections import defaultdict

def make_jobs(qids, qtext, chosen):
    text_of = dict(zip(qids, qtext))
    # Hai đoạn, mỗi đoạn tối đa 700 ký tự: một forward/doc nhưng vẫn có thêm bằng chứng;
    # tokenizer tiếp tục cắt cứng ở 512 token.
    return [(q, d, text_of[q], '\n[ĐOẠN LIÊN QUAN]\n'.join(CHUNK_TEXTS[ci][:700] for ci in cis))
            for q in qids for d, cis in chosen[q].items()]

def score_jobs(jobs, label, block=512):
    """Chia theo query round-robin; mỗi GPU xử lý các block nhỏ để log tiến độ."""
    q_order = list(dict.fromkeys(q for q, _, _, _ in jobs))
    owner = {q: i % len(DEVICES) for i, q in enumerate(q_order)}
    parts = [[x for x in jobs if owner[x[0]] == i] for i in range(len(DEVICES))]
    result, lock, done = {}, threading.Lock(), [0]

    def worker(dev, part):
        model = BGE_MODELS[dev]
        local = []
        for a in range(0, len(part), block):
            chunk = part[a:a+block]
            pairs = [[x[2], x[3]] for x in chunk]
            vals = model.compute_score(pairs, batch_size=4, normalize=False)
            vals = np.asarray(vals, dtype=float).reshape(-1)
            assert len(vals) == len(chunk) and np.isfinite(vals).all()
            local.extend((x[0], x[1], float(s)) for x, s in zip(chunk, vals))
            with lock:
                done[0] += len(chunk)
                print(f'[{label}] {done[0]}/{len(jobs)} pairs', flush=True)
        with lock:
            for q, d, s in local:
                result.setdefault(q, {})[d] = s

    t0 = time.time()
    threads = [threading.Thread(target=worker, args=(d, p), daemon=True)
               for d, p in zip(DEVICES, parts) if p]
    for t in threads: t.start()
    for t in threads: t.join()
    assert sum(map(len, result.values())) == len(jobs), 'Thiếu score do worker lỗi'
    print(f'[{label}] hoàn tất {(time.time()-t0)/60:.1f} phút')
    return result

VAL_SCORE = score_jobs(make_jobs(VAL_QIDS, val_text, VAL_CHUNK), 'validation')


100%|██████████| 128/128 [01:11<00:00,  1.79it/s]

[validation] 512/30000 pairs




pre tokenize: 100%|██████████| 128/128 [00:00<00:00, 361.13it/s]

100%|██████████| 128/128 [01:26<00:00,  1.48it/s]

[validation] 1024/30000 pairs



100%|██████████| 128/128 [01:19<00:00,  1.61it/s]

...

[validation] 29336/30000 pairs



100%|██████████| 128/128 [01:29<00:00,  1.43it/s]

[validation] 29848/30000 pairs



100%|██████████| 38/38 [00:27<00:00,  1.38it/s]

[validation] 30000/30000 pairs
[validation] hoàn tất 43.2 phút


In [18]:
#@title Chọn fusion trên validation; dừng nếu không thắng v3B
import pandas as pd

def rank_bge(candidates, scores, wr=None, k=60):
    out = {}
    for q, base in candidates.items():
        head = [d for d in base[:RERANK_TOP] if d in scores.get(q, {})]
        reranked = sorted(head, key=lambda d: -scores[q][d])
        if wr is None:
            out[q] = reranked + [d for d in base if d not in set(head)]
            continue
        fused = {}
        for i, d in enumerate(base):
            fused[d] = fused.get(d, 0.0) + 1.0 / (k+i+1)
        for i, d in enumerate(reranked):
            fused[d] = fused.get(d, 0.0) + wr / (k+i+1)
        out[q] = [d for d, _ in sorted(fused.items(), key=lambda z: -z[1])]
    return out

rows, PREDS = [], {}
for name, wr in [('replace', None), ('rrf_1', 1.0), ('rrf_2', 2.0),
                 ('rrf_3', 3.0), ('rrf_5', 5.0), ('rrf_8', 8.0)]:
    pred = rank_bge(dense_val, VAL_SCORE, wr=wr)
    met = evaluate(pred, {q: GOLD[q] for q in VAL_QIDS})
    PREDS[name] = pred
    rows.append({'mode': name, 'wr': wr, **met})

VAL_RESULTS = pd.DataFrame(rows).sort_values(
    ['OFFICIAL_recall', 'OFFICIAL_precision'], ascending=False).reset_index(drop=True)
display(VAL_RESULTS)
best = VAL_RESULTS.iloc[0]
BEST_MODE = str(best['mode'])
BEST_WR = None if BEST_MODE == 'replace' else float(best['wr'])

V3B_VAL = 0.9585
MIN_GAIN = 0.0010
print('v3B validation =', V3B_VAL)
print('v5 best =', BEST_MODE, '| recall =', float(best.OFFICIAL_recall),
      '| precision =', float(best.OFFICIAL_precision))
assert float(best.OFFICIAL_recall) >= V3B_VAL + MIN_GAIN, (
    'BGE-Gemma không thắng v3B đủ rõ trên validation. Notebook chủ động DỪNG trước public; '
    'hãy tiếp tục dùng submission v3B, không nộp v5.')
print('PASS VALIDATION GATE -> được phép chạy public')


,mode,wr,recall@5,recall@10,recall@20,recall@30,recall@50,recall@100,OFFICIAL_recall,OFFICIAL_precision
0,rrf_1,1.0,0.909583,0.960500,0.980167,0.982667,0.984917,0.987917,0.909583,0.1952
1,rrf_2,2.0,0.892833,0.943250,0.978833,0.982667,0.984917,0.987917,0.892833,0.1916
2,rrf_3,3.0,0.877583,0.934750,0.978333,0.982667,0.984917,0.987917,0.877583,0.1884
3,rrf_5,5.0,0.864583,0.925417,0.976333,0.982667,0.984917,0.987917,0.864583,0.1858
4,rrf_8,8.0,0.853250,0.918667,0.974583,0.982667,0.984917,0.987917,0.853250,0.1830
5,replace,NaN,0.842750,0.912667,0.968250,0.982667,0.984917,0.987917,0.842750,0.1806


v3B validation = 0.9585
v5 best = rrf_1 | recall = 0.9095833333333334 | precision = 0.19520000000000004


AssertionError: BGE-Gemma không thắng v3B đủ rõ trên validation. Notebook chủ động DỪNG trước public; hãy tiếp tục dùng submission v3B, không nộp v5.

In [ ]:
#@title Chạy public bằng đúng cấu hình đã thắng validation
PUB_SCORE = score_jobs(make_jobs(pub_qids, pub_text, PUB_CHUNK), 'public')
pub_preds = rank_bge(pub_dense, PUB_SCORE, wr=BEST_WR)
print('public predictions:', len(pub_preds), '| mode:', BEST_MODE)


In [ ]:
#@title Validate định dạng và tạo ZIP nộp bài
import json, zipfile, os, collections

def build_submission(preds, all_qids):
    corpus_ids = set(map(str, DOC_IDS))
    fallback = list(map(str, DOC_IDS[:TOPK_SUBMIT]))
    sub = {}
    for q in all_qids:
        ans, seen = [], set()
        for d in list(preds.get(q, [])) + fallback:
            d = str(d)
            if d in corpus_ids and d not in seen:
                seen.add(d); ans.append(d)
            if len(ans) == TOPK_SUBMIT:
                break
        sub[str(q)] = {'answer': ans}
    return sub

sub = build_submission(pub_preds, pub_qids)
assert set(sub) == set(map(str, pub_qids)), 'Thiếu/thừa query id'
for q, v in sub.items():
    a = v['answer']
    assert len(a) == 5 and len(set(a)) == 5, (q, a)
    assert all(isinstance(x, str) for x in a), (q, a)

out_json = '/kaggle/working/submission.json' if ON_KAGGLE else f'{ART}/submission.json'
out_zip = ('/kaggle/working/submission_v5_bge_gemma_3b.zip' if ON_KAGGLE
           else f'{ART}/submission_v5_bge_gemma_3b.zip')
with open(out_json, 'w', encoding='utf-8') as f:
    json.dump(sub, f, ensure_ascii=False)
with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(out_json, arcname='submission.json')
with zipfile.ZipFile(out_zip) as z:
    assert z.namelist() == ['submission.json']
print('OK ->', out_zip)
print('phân bố số id:', collections.Counter(len(v['answer']) for v in sub.values()))


## Cách chạy

1. Add Input dataset `kitnehi1211/v3b-finetune-reranker-improve` cùng publictest và scoring.
2. Đăng ký/kiểm tra model `BAAI/bge-reranker-v2-gemma` với BTC.
3. Sau khi được chấp nhận, đổi `MODEL_APPROVED_BY_BTC=True`. Giữ `RUN_PHASE=2`.
4. Bật Internet, chọn GPU T4 x2 và Save & Run All đúng một lần.

Nếu validation không đạt ít nhất 0.9595, notebook dừng trước public và không tạo ZIP.
